In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_small_q4/checkpoints/post_cell_1_try_3.pickle

In [4]:
%%RecordEvent
%%time
### cell 2 ###

# Filter lineitem where commit date < receipt date on GPU
lsel = lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE
# Keep only the key for the join
flineitem = lineitem[lsel][["L_ORDERKEY"]]

# Apply date filters using string literals to avoid CPU Timestamp creation
osel = (orders.O_ORDERDATE < "1993-11-01") & (orders.O_ORDERDATE >= "1993-08-01")
# Keep only the columns needed downstream
forders = orders[osel][["O_ORDERKEY", "O_ORDERPRIORITY"]]

# Perform an inner join on the GPU rather than isin-based filtering
jn = forders.merge(flineitem, left_on="O_ORDERKEY", right_on="L_ORDERKEY")

# Group and count entirely on the GPU
total = jn.groupby("O_ORDERPRIORITY", as_index=False).O_ORDERKEY.count()

CPU times: user 62.5 ms, sys: 57.1 ms, total: 120 ms
Wall time: 122 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_small_q4/checkpoints/post_cell_2_try_0.pickle